# Demo 2: Predictive Modeling with Sleep Monitoring Data

In this demo, we'll use the Multilevel Monitoring of Activity and Sleep (MMASH) dataset to build predictive models for sleep quality. We'll learn how to:
- Work with multivariate health time series data
- Engineer features from temporal health data
- Build and evaluate regression models
- Compare different cohorts and conditions

## 1. Setting Up and Loading the Data

First, let's import the necessary libraries and download the dataset from PhysioNet.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from urllib.request import urlretrieve
import os

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Create a directory for data if it doesn't exist
os.makedirs('data', exist_ok=True)

In [ ]:
# Download data from PhysioNet
base_url = "https://physionet.org/files/mmash/1.0.0/"

# Download info.csv file
info_url = base_url + "info.csv"
info_file = "data/mmash_info.csv"
urlretrieve(info_url, info_file)

# Download sleep data for a few subjects
subjects = ['1', '2', '3']
sleep_files = {}
activity_files = {}

for subject in subjects:
    # Sleep data
    sleep_url = base_url + f"user_{subject}/sleep.csv"
    sleep_local = f"data/sleep_{subject}.csv"
    urlretrieve(sleep_url, sleep_local)
    sleep_files[subject] = sleep_local
    
    # Activity data
    activity_url = base_url + f"user_{subject}/Activity.csv"
    activity_local = f"data/activity_{subject}.csv"
    urlretrieve(activity_url, activity_local)
    activity_files[subject] = activity_local

print(f"Downloaded data for {len(subjects)} subjects")

Now let's load the participant information and understand what's in the dataset.

In [ ]:
# Load participant information
info = pd.read_csv(info_file)

# Display the first few rows
print("Participant information:")
info.head()

In [ ]:
# Load sleep data for the first subject
subject_id = subjects[0]
sleep_data = pd.read_csv(sleep_files[subject_id])

# Display the first few rows
print(f"Sleep data for subject {subject_id}:")
sleep_data.head()

In [ ]:
# Load activity data for the first subject
activity_data = pd.read_csv(activity_files[subject_id])

# Display the first few rows
print(f"Activity data for subject {subject_id}:")
activity_data.head()

## 2. Preparing and Exploring the Data

Let's combine sleep and activity data for all subjects and explore the relationships.

In [ ]:
# Function to process data for a single subject
def process_subject_data(subject_id):
    # Load sleep data
    sleep = pd.read_csv(sleep_files[subject_id])
    
    # Load activity data
    activity = pd.read_csv(activity_files[subject_id])
    
    # Get participant info
    subject_info = info[info['user'] == int(subject_id)].iloc[0]
    
    # Aggregate activity data by day
    # Convert timestamp to datetime if needed
    if 'timestamp' in activity.columns:
        activity['timestamp'] = pd.to_datetime(activity['timestamp'])
        activity['date'] = activity['timestamp'].dt.date
    else:
        # If no timestamp, assume it's already daily data
        activity['date'] = pd.to_datetime(activity.index).date
    
    # Calculate daily activity metrics
    daily_activity = activity.groupby('date').agg({
        'steps': 'sum',
        'calories': 'sum'
    }).reset_index()
    
    # Merge sleep and activity data
    # Assuming sleep data has a date column or can be linked to activity dates
    if 'date' not in sleep.columns and 'night' in sleep.columns:
        # If sleep data has 'night' instead of 'date', create a date column
        # This is a simplification - in real data, you'd need to align dates properly
        sleep['date'] = pd.to_datetime(daily_activity['date'].iloc[0]) + pd.to_timedelta(sleep['night'] - 1, unit='d')
    
    # Merge data
    merged_data = pd.merge(sleep, daily_activity, on='date', how='inner')
    
    # Add subject info
    merged_data['subject_id'] = subject_id
    merged_data['age'] = subject_info['age']
    merged_data['gender'] = subject_info['gender']
    merged_data['BMI'] = subject_info['BMI']
    
    return merged_data

# Process data for all subjects
all_data = []
for subject in subjects:
    try:
        subject_data = process_subject_data(subject)
        all_data.append(subject_data)
    except Exception as e:
        print(f"Error processing subject {subject}: {e}")

# Combine all data
combined_data = pd.concat(all_data, ignore_index=True)

# Display the combined dataset
print("Combined sleep and activity data:")
combined_data.head()

In [ ]:
# Explore relationships between variables
plt.figure(figsize=(12, 10))

# Sleep quality vs. steps
plt.subplot(2, 2, 1)
sns.scatterplot(x='steps', y='sleep_quality', hue='subject_id', data=combined_data)
plt.title('Sleep Quality vs. Steps')
plt.xlabel('Daily Steps')
plt.ylabel('Sleep Quality')

# Sleep quality vs. calories
plt.subplot(2, 2, 2)
sns.scatterplot(x='calories', y='sleep_quality', hue='subject_id', data=combined_data)
plt.title('Sleep Quality vs. Calories Burned')
plt.xlabel('Daily Calories Burned')
plt.ylabel('Sleep Quality')

# Sleep quality vs. BMI
plt.subplot(2, 2, 3)
sns.scatterplot(x='BMI', y='sleep_quality', hue='gender', data=combined_data)
plt.title('Sleep Quality vs. BMI')
plt.xlabel('BMI')
plt.ylabel('Sleep Quality')

# Correlation matrix
plt.subplot(2, 2, 4)
numeric_cols = combined_data.select_dtypes(include=[np.number]).columns.tolist()
correlation_matrix = combined_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

Let's create features that might help predict sleep quality.

In [ ]:
# Create a copy of the data for feature engineering
data_with_features = combined_data.copy()

# 1. Create time-based features if date is available
if 'date' in data_with_features.columns:
    data_with_features['date'] = pd.to_datetime(data_with_features['date'])
    data_with_features['day_of_week'] = data_with_features['date'].dt.dayofweek
    data_with_features['is_weekend'] = data_with_features['day_of_week'].isin([5, 6]).astype(int)

# 2. Create activity intensity features
if 'steps' in data_with_features.columns:
    # Categorize activity level based on steps
    data_with_features['activity_level'] = pd.cut(
        data_with_features['steps'],
        bins=[0, 5000, 10000, 15000, float('inf')],
        labels=['Low', 'Moderate', 'High', 'Very High']
    )

# 3. Create BMI category
if 'BMI' in data_with_features.columns:
    data_with_features['bmi_category'] = pd.cut(
        data_with_features['BMI'],
        bins=[0, 18.5, 25, 30, float('inf')],
        labels=['Underweight', 'Normal', 'Overweight', 'Obese']
    )

# 4. Create interaction features
if 'steps' in data_with_features.columns and 'calories' in data_with_features.columns:
    data_with_features['steps_per_calorie'] = data_with_features['steps'] / data_with_features['calories']

# Display the data with new features
print("Data with engineered features:")
data_with_features.head()

In [ ]:
# Convert categorical features to dummy variables
categorical_cols = ['gender', 'activity_level', 'bmi_category']
categorical_cols = [col for col in categorical_cols if col in data_with_features.columns]

if categorical_cols:
    data_encoded = pd.get_dummies(data_with_features, columns=categorical_cols, drop_first=True)
else:
    data_encoded = data_with_features.copy()

# Drop non-numeric columns and columns we don't want to use as features
cols_to_drop = ['date', 'subject_id']
cols_to_drop = [col for col in cols_to_drop if col in data_encoded.columns]
data_encoded = data_encoded.drop(columns=cols_to_drop)

# Drop rows with missing values after feature engineering
data_encoded = data_encoded.dropna()

print(f"Final dataset shape: {data_encoded.shape}")
print(f"Features: {data_encoded.columns.tolist()}")

## 4. Building Regression Models

Now let's build models to predict sleep quality based on our features.

In [ ]:
# Prepare data for modeling
X = data_encoded.drop(columns=['sleep_quality'])
y = data_encoded['sleep_quality']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [ ]:
# Function to evaluate regression models
def evaluate_model(model, X_train, X_test, y_train, y_test):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Print metrics
    print(f"Training MAE: {train_mae:.4f}")
    print(f"Testing MAE: {test_mae:.4f}")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Testing RMSE: {test_rmse:.4f}")
    print(f"Training R²: {train_r2:.4f}")
    print(f"Testing R²: {test_r2:.4f}")
    
    return model, y_pred_test

In [ ]:
# 1. Linear Regression
print("Linear Regression Model:")
lr_model = LinearRegression()
lr_model, lr_preds = evaluate_model(lr_model, X_train, X_test, y_train, y_test)

# Print coefficients
print("\nLinear Regression Coefficients:")
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print(coef_df.head(10))

In [ ]:
# 2. Random Forest Regression
print("Random Forest Regression Model:")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model, rf_preds = evaluate_model(rf_model, X_train, X_test, y_train, y_test)

# Print feature importances
print("\nRandom Forest Feature Importances:")
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
importance_df = importance_df.sort_values('Importance', ascending=False)
print(importance_df.head(10))

In [ ]:
# Visualize predictions vs. actual values
plt.figure(figsize=(12, 6))

# Linear Regression
plt.subplot(1, 2, 1)
plt.scatter(y_test, lr_preds, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.title('Linear Regression: Actual vs. Predicted')
plt.xlabel('Actual Sleep Quality')
plt.ylabel('Predicted Sleep Quality')

# Random Forest
plt.subplot(1, 2, 2)
plt.scatter(y_test, rf_preds, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.title('Random Forest: Actual vs. Predicted')
plt.xlabel('Actual Sleep Quality')
plt.ylabel('Predicted Sleep Quality')

plt.tight_layout()
plt.show()

## 5. Comparing Different Cohorts

Let's compare how the relationship between activity and sleep quality differs across different cohorts.

In [ ]:
# Create cohorts based on BMI
if 'BMI' in combined_data.columns:
    combined_data['bmi_group'] = pd.cut(
        combined_data['BMI'],
        bins=[0, 25, float('inf')],
        labels=['Normal/Underweight', 'Overweight/Obese']
    )

# Create cohorts based on activity level
if 'steps' in combined_data.columns:
    combined_data['activity_group'] = pd.cut(
        combined_data['steps'],
        bins=[0, 7500, float('inf')],
        labels=['Low Activity', 'High Activity']
    )

# Compare sleep quality across cohorts
plt.figure(figsize=(12, 5))

# By BMI group
if 'bmi_group' in combined_data.columns:
    plt.subplot(1, 2, 1)
    sns.boxplot(x='bmi_group', y='sleep_quality', data=combined_data)
    plt.title('Sleep Quality by BMI Group')
    plt.xlabel('BMI Group')
    plt.ylabel('Sleep Quality')

# By activity group
if 'activity_group' in combined_data.columns:
    plt.subplot(1, 2, 2)
    sns.boxplot(x='activity_group', y='sleep_quality', data=combined_data)
    plt.title('Sleep Quality by Activity Group')
    plt.xlabel('Activity Group')
    plt.ylabel('Sleep Quality')

plt.tight_layout()
plt.show()

## 6. Summary and Discussion

In this demo, we've explored sleep monitoring data and built predictive models. Here's what we've learned:

1. **Data Integration**: We combined sleep data with activity data and participant information to create a comprehensive dataset.

2. **Feature Engineering**: We created various features from the raw data, including time-based features, activity intensity categories, and interaction features.

3. **Regression Modeling**: We built and compared linear regression and random forest models to predict sleep quality.

4. **Cohort Analysis**: We compared sleep quality across different cohorts based on BMI and activity level.

### Key Insights:

- Physical activity appears to have a relationship with sleep quality, though the relationship may not be linear.
- Random Forest models generally outperform Linear Regression for this task, suggesting non-linear relationships.
- Different cohorts (e.g., by BMI or activity level) may show different patterns in sleep quality.

### Next Steps:

- Collect more data to improve model robustness.
- Explore more advanced time series features, such as lag features across multiple days.
- Consider time series cross-validation for more accurate model evaluation.
- Investigate the causal relationship between activity and sleep quality.